In [ ]:
!pip install lpips
!pip install torchinfo
!pip install torch-fidelity

In [ ]:
import sys
repo_path = "/kaggle/input/datasets/dahyuntw/defectfill-model/DefectFill Repo/DefectFill"

sys.path.append(repo_path)

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from data_loader import get_data_loaders
from model import DefectFillModel
import json
import cv2
from PIL import Image
from torchvision.utils import save_image
from torchvision import transforms
from datetime import datetime
import argparse
import os
from torchinfo import summary
from torch.utils.tensorboard import SummaryWriter
from utils import save_checkpoint, load_checkpoint
from diffusers import DDPMScheduler
from tqdm import tqdm
import numpy as np
from pathlib import Path

## 1. Configuration

In [ ]:
DATA_DIR = "/kaggle/input/datasets/dahyuntw/curated-mvtec/curated_mvtec"
OUTPUT_DIR = "/kaggle/working/"
OBJ_CLASS = "bottle"
DEFECT_TYPE = None # broken_large OR None -> load all
HF_MODEL_ID = "sd2-community/stable-diffusion-2-inpainting"

BATCH_SIZE = 2
RESUME_FROM = None # Path to the checkpoint file
LORA_RANK = 8
LORA_ALPHA = 16 # Effect of LoRA on fine-tuning: high alpha -> high effect
UNET_LR = 2e-4
TEXT_ENC_LR = 4e-5
WARM_UP_STEPS = 100
MAX_TRAIN_STEPS = 500 # total_steps
GRAD_ACCUM_STEPS = 2

SAVE_STEPS = 250
LAMBDA_DEFECT = 0.5
LAMBDA_OBJ = 0.2
LAMBDA_ATTN = 0.05
ALPHA = 0.3 # Background weight for object branch

# Ablation Study
ABLATE_DEFECT_LOSS = False  # Set True to zero out defect loss
ABLATE_OBJ_LOSS    = False  # Set True to zero out object loss  
ABLATE_ATTN_LOSS   = False  # Set True to zero out attention loss

## Setup

In [ ]:
checkpoints_dir = os.path.join(OUTPUT_DIR, "checkpoints")
tensorboard_dir = os.path.join(OUTPUT_DIR, "tensorboard")
os.makedirs(checkpoints_dir, exist_ok=True)
os.makedirs(tensorboard_dir, exist_ok=True)

## Load data

In [ ]:
train_loader, test_loader = get_data_loaders(
    root_dir=DATA_DIR,
    object_class=OBJ_CLASS,
    batch_size=BATCH_SIZE,
    defect_type=DEFECT_TYPE
)

## Initialize model & optimizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if device.type == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")

In [ ]:
defect_root = Path(
    os.path.join(DATA_DIR, OBJ_CLASS, "train" ,"defective")
)

defect_dict = {
    f'{OBJ_CLASS}_{folder.name}': f'{folder.name}'
    for folder in defect_root.iterdir()
    if folder.is_dir()
}
print(defect_dict)

In [ ]:
model = DefectFillModel(
    device=device,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    defect_token_map=defect_dict
)

In [ ]:
text_encoder_params = [p for n, p in model.pipeline.text_encoder.named_parameters() if "lora" in n]
unet_params         = [p for n, p in model.pipeline.unet.named_parameters()         if "lora" in n]

optimizer = AdamW([
    {"params": text_encoder_params, "lr": TEXT_ENC_LR},
    {"params": unet_params,         "lr": UNET_LR},
])

base_lrs = [TEXT_ENC_LR, UNET_LR] # Save the original LR for warm-up calculations

In [ ]:
summary(model)

There're just about 3,000 trainable params out of more than 14M params, saving major training effort.

## Noise Scheduler

In [ ]:
# Determines how and when noise is removed step-by-step
noise_scheduler = DDPMScheduler.from_pretrained(HF_MODEL_ID, subfolder="scheduler")

## Continue from the last model

In [ ]:
start_step = 0
if RESUME_FROM:
    start_step = load_checkpoint(model, optimizer, RESUME_FROM)
    print(f"Resumed from step {start_step}")

# 2. Training loop

In [ ]:
writer = SummaryWriter(tensorboard_dir)

In [ ]:
model.pipeline.unet.train()
model.pipeline.text_encoder.train()

In [ ]:
global_step    = start_step
accum_step     = 0
total_steps    = MAX_TRAIN_STEPS # total_steps = (batch_size / gradient_accumulation_steps) * num_epochs
progress_bar = tqdm(total=total_steps - start_step, desc="Training")

while global_step < total_steps:
    for batch in train_loader:
        if global_step >= total_steps:
            break

        images      = batch["image"].to(device, dtype=torch.float16)
        masks       = batch["mask"].to(device, dtype=torch.float16)
        backgrounds = batch["background"].to(device, dtype=torch.float16)
        adj_masks   = batch["adjusted_mask"].to(device, dtype=torch.float16)
        is_defect   = batch["is_defect"]

        defect_idx = torch.nonzero(is_defect).squeeze(1)
        if len(defect_idx) == 0:
            print("No defect in this batch")
            continue # No defect, skip

        d_images = images[defect_idx]
        d_masks = masks[defect_idx]
        d_bgs    = backgrounds[defect_idx]
        d_adj = adj_masks[defect_idx]
        obj_cls  = [batch["object_class"][i] for i in defect_idx] # bottle, transistor, ...
        
        defect_types = []
        for i in defect_idx:
            img_path = train_loader.dataset.images[i] if i < len(train_loader.dataset.images) else ""
            parts = img_path.split(os.sep)
            for j, part in enumerate(parts):
                if part == "defective" and j+1 < len(parts):
                    defect_types.append(parts[j+1]); break
            else:
                defect_types.append("defect")

        # LR warmup
        if global_step < WARM_UP_STEPS:
            scale = (global_step + 1) / WARM_UP_STEPS
            for i, pg in enumerate(optimizer.param_groups):
                pg["lr"] = base_lrs[i] * scale
        # Reset attention maps
        if hasattr(model, "attention_maps"): 
            model.attention_maps = {}

        # ---- Phase 1: defect branch -> How the placeholder_token looks like in context
        # placeholder_token -> encodes model's understanding of the defect through textual inversion
        # i.e. textual inversion associates defects -> a textually latent representation rather than mapping 
        # it directly to a new word
        # By the end of the training process placeholder_token is linked to a concept
        d_prompts    = [f"A photo of {model.get_token_for(c, d)}" for c, d in zip(obj_cls, defect_types)]

        text_emb     = model.get_text_embeddings(d_prompts, enable_grad=True)

        model.pipeline.vae.to(device)
        # Latent space processing: Converts and image from pixel space -> latent space
        with torch.no_grad():
            latents = model.pipeline.vae.encode(d_images).latent_dist.sample()
            latents = latents * model.pipeline.vae.config.scaling_factor
            
        model.pipeline.vae.to("cpu")
        torch.cuda.empty_cache()
        
        noise     = torch.randn_like(latents) # Generate random noise
        # Small timestep -> Less noisy and vice versa
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                  (latents.shape[0],), device=device)
        noisy_lat = noise_scheduler.add_noise(latents, noise, timesteps) # z_t: Noise added to the clean latent

        if d_masks.dim() == 3: 
            d_masks = d_masks.unsqueeze(1)
            
        model.pipeline.vae.to(device)
        with torch.no_grad():
            masked_lat = model.pipeline.vae.encode(
                d_images * (1 - d_masks)).latent_dist.sample()
            masked_lat = masked_lat * model.pipeline.vae.config.scaling_factor
        model.pipeline.vae.to("cpu")
        torch.cuda.empty_cache()
        
        mask_lat = F.interpolate(d_masks, size=latents.shape[2:], mode='bilinear', align_corners=False)
  
        # Forward pass (9-channel input)
        # Generateion: predicts noise -> inference is reverse diffusion
        outputs  = model(noisy_latents=noisy_lat, 
                         masked_image_latents=masked_lat,
                         mask_latents=mask_lat, 
                         timesteps=timesteps,
                         encoder_hidden_states=text_emb)

        defect_loss = model.compute_defect_loss(outputs["noise_pred"], noise, mask_lat)
        attn_loss   = outputs.get("attention_loss", torch.tensor(0.0, device=device))

        # ---- Phase 2: object branch -> How does the normal object look like (without defects)
        rand_masks = torch.zeros_like(d_images[:, :1])
        for i in range(rand_masks.shape[0]):
            m = rand_masks[i, 0]; h, w = m.shape
            for _ in range(30): # 30 random boxes
                rh = torch.randint(int(min(h,w)*.03), max(int(min(h,w)*.03)+1, int(min(h,w)*.25)), (1,)).item()
                rw = torch.randint(int(min(h,w)*.03), max(int(min(h,w)*.03)+1, int(min(h,w)*.25)), (1,)).item()
                y = torch.randint(0, max(1, h-rh), (1,)).item()
                x = torch.randint(0, max(1, w-rw), (1,)).item()
                m[y:y+rh, x:x+rw] = 1.0

        obj_prompts = [f"A {c} with {model.get_token_for(c, d)}" for c, d in zip(obj_cls, defect_types)]
        obj_emb     = model.get_text_embeddings(obj_prompts, enable_grad=True)
        obj_noise   = torch.randn_like(latents)
        obj_ts      = torch.randint(0, noise_scheduler.config.num_train_timesteps,
                                    (latents.shape[0],), device=device)
        obj_noisy   = noise_scheduler.add_noise(latents, obj_noise, obj_ts)

        model.pipeline.vae.to(device)
        with torch.no_grad():
            rm_lat = model.pipeline.vae.encode(
                d_images * (1 - rand_masks)).latent_dist.sample()
            rm_lat = rm_lat * model.pipeline.vae.config.scaling_factor
        model.pipeline.vae.to("cpu")
        torch.cuda.empty_cache()
        
        rm_mask_lat = F.interpolate(rand_masks, size=latents.shape[2:], mode='bilinear', align_corners=False)
        obj_out     = model(noisy_latents=obj_noisy, masked_image_latents=rm_lat,
                            mask_latents=rm_mask_lat, timesteps=obj_ts,
                            encoder_hidden_states=obj_emb)
        obj_loss    = model.compute_object_loss(obj_out["noise_pred"], obj_noise,
                                                rm_mask_lat, alpha=ALPHA)

        # Loss calculation with ablation gates
        effective_defect = defect_loss * (0.0 if ABLATE_DEFECT_LOSS else LAMBDA_DEFECT)
        effective_obj    = obj_loss    * (0.0 if ABLATE_OBJ_LOSS    else LAMBDA_OBJ)
        effective_attn   = attn_loss   * (0.0 if ABLATE_ATTN_LOSS   else LAMBDA_ATTN)
        
        total_loss = (effective_defect + effective_obj + effective_attn) / GRAD_ACCUM_STEPS

        writer.add_scalar("Debug/noise_pred_mean", outputs["noise_pred"].float().abs().mean().item(), global_step)
        writer.add_scalar("Debug/noise_mean",      noise.float().abs().mean().item(),                 global_step)
        writer.add_scalar("Debug/latent_mean",     latents.float().abs().mean().item(), global_step)

        if torch.isnan(total_loss):
            print(f"NaN at step {global_step} - skipping")
            print(f"  defect_loss: {defect_loss.item()}")
            print(f"  obj_loss:    {obj_loss.item()}")
            print(f"  attn_loss:   {attn_loss.item()}")
            optimizer.zero_grad()
            continue

        total_loss.backward()
        accum_step += 1

        if accum_step >= GRAD_ACCUM_STEPS:
            optimizer.step()
            optimizer.zero_grad()
            accum_step = 0
            progress_bar.update(1)
            global_step += 1
 
            writer.add_scalar("Loss/Defect",  defect_loss.item(), global_step)
            writer.add_scalar("Loss/Object",   obj_loss.item(),   global_step)
            writer.add_scalar("Loss/Attention", attn_loss.item(), global_step)
            writer.add_scalar("Loss/Total", total_loss.item()*GRAD_ACCUM_STEPS, global_step)
           
            # Log which config is running
            if global_step == 1:
                ablation_tag = "_".join([
                    "full" if not any([ABLATE_DEFECT_LOSS, ABLATE_OBJ_LOSS, ABLATE_ATTN_LOSS])
                    else "",
                    "" if ABLATE_DEFECT_LOSS else "defect",
                    "" if ABLATE_OBJ_LOSS    else "obj",
                    "" if ABLATE_ATTN_LOSS   else "attn",
                ]).strip("_")
                print(f"[Ablation] Running configuration: {ablation_tag}")
                writer.add_text("Ablation/Config", ablation_tag, global_step)
            
            if global_step % 10 == 0:
                for i, pg in enumerate(optimizer.param_groups):
                    writer.add_scalar(f"LR/group{i}", pg["lr"], global_step)

            if global_step % SAVE_STEPS == 0 or global_step == total_steps:
                # Save the model every SAVE_STEPS
                ckpt = os.path.join(checkpoints_dir, f"checkpoint_{global_step}.pth")
                save_checkpoint(model, optimizer, global_step, ckpt)

# Save final model
final_checkpoint_path = os.path.join(checkpoints_dir, "checkpoint_final.pth")
save_checkpoint(model, optimizer, global_step, final_checkpoint_path)
writer.close()

# 3. Inference

In [ ]:
from inference import calculate_generation_plan, count_available_resources
import matplotlib.pyplot as plt

In [ ]:
def show_tensors(clean, sample, mask):
    """Display 3 tensors inline in notebook. Expects [-1,1] for images, [0,1] for mask."""
    def to_np(t):
        t = t.squeeze(0).float().cpu()
        if t.shape[0] == 3:          # image [-1, 1] → [0, 1]
            t = (t + 1) / 2
        else:                         # mask, already [0, 1]
            t = t.repeat(3, 1, 1)    # grayscale → RGB for display
        return t.permute(1, 2, 0).clamp(0, 1).numpy()

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    titles = ["Clean (reference)", "Generated sample", "Mask"]
    
    for ax, img, title in zip(axes, [clean, sample, mask], titles):
        ax.imshow(to_np(img))
        ax.set_title(title)
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()



In [ ]:
def flush():
    """VRAM cleanup between generations."""
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

def offload_to_cpu(model):
    """Move text encoder + VAE to CPU when not needed."""
    model.pipeline.text_encoder.to("cpu")
    model.pipeline.vae.to("cpu")
    torch.cuda.empty_cache()

def reload_to_gpu(model, device, dtype):
    """Bring them back just before generation."""
    model.pipeline.text_encoder.to(device, dtype=dtype)
    model.pipeline.vae.to(device, dtype=dtype)


def paste_back(original_np, generated_crop_tensor, x1, y1, x2, y2, base_size=512):
    """Paste the generated 512x512 crop back into the original image."""
    # Convert tensor to numpy [0, 255]
    gen_np = (generated_crop_tensor.float().squeeze(0).permute(1,2,0).cpu().numpy() + 1) / 2
    gen_np = (gen_np * 255).clip(0, 255).astype(np.uint8)

    crop_h = y2 - y1
    crop_w = x2 - x1

    # Resize back to original crop size (only if it was resized during smart crop)
    if crop_h != base_size or crop_w != base_size:
        gen_np = cv2.resize(gen_np, (crop_w, crop_h), interpolation=cv2.INTER_LINEAR)

    result = original_np.copy()
    result[y1:y2, x1:x2] = gen_np
    return result

In [ ]:
def smart_crop_dynamic(image, mask, base_size=512):
    """
    Adapted smart_crop
    Crops the image to fit the defect. 
    - If defect < 512: Crops 512x512 (No Resize).
    - If defect > 512: Crops square enclosing defect, then resizes to 512.
    """
    h, w = image.shape[:2]
    
    # Find the Bounding Box of the defect
    y_indices, x_indices = np.where(mask > 0)
    
    if len(y_indices) == 0:
        # No defect? Return center crop 512
        cy, cx = h // 2, w // 2
        crop_size = base_size
    else:
        min_y, max_y = np.min(y_indices), np.max(y_indices)
        min_x, max_x = np.min(x_indices), np.max(x_indices)
        
        defect_h = max_y - min_y
        defect_w = max_x - min_x
        
        # Center of the defect
        cy = min_y + defect_h // 2
        cx = min_x + defect_w // 2
        
        # Determine the Crop Size
        # We need a box big enough to hold the defect + some context padding
        # But at minimum, it must be 512.
        max_dim = max(defect_h, defect_w)
        padding = 50 # Add 50px context around edges if possible
        
        crop_size = max(base_size, max_dim + padding)
    
    # Calculate Crop Coordinates (Square Box)
    half_size = crop_size // 2
    x1 = cx - half_size
    y1 = cy - half_size
    x2 = x1 + crop_size
    y2 = y1 + crop_size
    
    # Handle Edge Cases (Shift box if it goes out of bounds)
    if x1 < 0: x2 -= x1; x1 = 0
    if y1 < 0: y2 -= y1; y1 = 0
    if x2 > w: x1 -= (x2 - w); x2 = w
    if y2 > h: y1 -= (y2 - h); y2 = h
    
    # Double check we didn't shrink below image dims (e.g. if image is smaller than crop_size)
    x1 = max(0, x1); y1 = max(0, y1)
    x2 = min(w, x2); y2 = min(h, y2)

    # Perform the Crop
    crop_img = image[y1:y2, x1:x2]
    crop_mask = mask[y1:y2, x1:x2]
    
    # Resize ONLY if the crop is larger than 512
    # (If crop_size was 512, this does nothing. If it was 570, it shrinks slightly.)
    if crop_img.shape[0] != base_size or crop_img.shape[1] != base_size:
        crop_img = cv2.resize(crop_img, (base_size, base_size), interpolation=cv2.INTER_AREA)
        # Use NEAREST for mask to keep edges sharp
        crop_mask = cv2.resize(crop_mask, (base_size, base_size), interpolation=cv2.INTER_NEAREST)
        
    return crop_img, crop_mask, x1, y1, x2, y2

def compute_spatial_lpips(lpips_model, img1, img2, mask, smooth_boundary=True):
    """
    Calculates the Perceptual Distance specifically within the masked region using Spatial LPIPS
    
    Args:
        lpips_model: LPIPS model instance initialized with spatial=True
        img1: Reference image [B, 3, H, W], range [-1, 1]
        img2: Comparison image [B, 3, H, W], range [-1, 1]
        mask: Defect mask [B, 1, H, W], range [0, 1]
        smooth_boundary: Whether to blur the mask edges to avoid boundary artifacts
    
    Returns:
        lpips_score: LPIPS score for the masked region (scalar)
    """
    # Type cast to float32 (lpips_map is float32, otw mask_sum = inf)
    img1 = img1.float()
    img2 = img2.float()
    mask = mask.float()
    
    # 1. Compute spatial LPIPS map (pixel-wise perceptual distance)
    lpips_map = lpips_model(img1, img2)  # Output shape: [B, 1, H', W']
    # 2. Resize the mask to match the LPIPS output resolution
    mask_resized = F.interpolate(
        mask, 
        size=lpips_map.shape[-2:], 
        mode='bilinear',
        align_corners=False
    )
    
    # 3. Optional: Smooth boundary edges (Gaussian-like blur via AvgPool)
    if smooth_boundary:
        mask_smoothed = F.avg_pool2d(
            F.pad(mask_resized, (2, 2, 2, 2), mode='replicate'),
            kernel_size=5, stride=1
        )
    else:
        mask_smoothed = mask_resized
    
    # 4. Mask-weighted summation
    weighted_sum = (lpips_map * mask_smoothed).sum(dim=(2, 3))
    mask_sum = mask_smoothed.sum(dim=(2, 3)) + 1e-8

    # 5. Return normalized LPIPS score
    return (weighted_sum / mask_sum).mean()

In [ ]:
def generate_one_sample(model, clean_image, mask, prompt, steps, guidance_scale, seed, device, dtype):
    """
    Generates exactly ONE sample to avoid stacking tensors in VRAM.
    """
    generator = torch.Generator(device=device).manual_seed(seed)

    with torch.no_grad():
        sample = model.generate(
            image=clean_image,
            mask=mask,
            prompt=prompt,
            num_inference_steps=steps,
            guidance_scale=guidance_scale,
            generator=generator,
        )  # returns [0, 1]

    return (sample * 2.0) - 1.0  # convert to [-1, 1]
    
def select_best_sample(model, clean_image, mask, prompt,
                       num_samples, steps, guidance_scale,
                       device, dtype):
    """
    Generates num_samples ONE AT A TIME and scores each with LPIPS.
    Keeps only the best
    """
    best_sample, best_score = None, -float("inf")

    _, _, h, w = clean_image.shape
    mask_resized = mask if mask.shape[-2:] == (h, w) else \
                   torch.nn.functional.interpolate(mask, size=(h, w), mode='bilinear')

    for i in range(num_samples):
        sample = generate_one_sample(
            model, clean_image, mask, prompt, steps, guidance_scale, seed=i, device=device, dtype=dtype
        )

        # Resize if sample size # clean image size
        if sample.shape[-2:] != (h, w):
            sample = torch.nn.functional.interpolate(sample, size=(h, w), mode='bilinear')

        # show_tensors(clean_image, sample, mask_resized)
        
        # Score with LPIPS (DL model judging the similarity of two images) (single sample, no batch needed)
        score = compute_spatial_lpips(
            model.lpips_model, clean_image, sample, mask_resized
        )

        score_val = score.item() if torch.is_tensor(score) else float(score)

        print(f"\tSample {i+1}/{num_samples} — LPIPS: {score_val:.4f}")

        if score_val > best_score:
            best_score  = score_val
            best_sample = sample.clone()

        del sample
        flush()

    return best_sample, best_score

def inference(args):
    dtype  = torch.float16

    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32       = True

    # ---------- Load model ----------
    # Load defect_token_map 
    defect_token_map = {}
    if args.checkpoint:
        meta = torch.load(args.checkpoint, map_location="cpu")
        defect_token_map = meta.get("defect_token_map", {})

    # Init model
    model = DefectFillModel(
        device=device,
        lora_rank=args.lora_rank,
        lora_alpha=args.lora_alpha,
        defect_token_map=defect_token_map
    )
    model.pipeline.vae.to(dtype=dtype)
    
    if args.checkpoint:
        load_checkpoint(model, None, args.checkpoint)
        print(f"Loaded checkpoint: {args.checkpoint}")
        
    model.pipeline.unet.eval()
    model.pipeline.text_encoder.eval()

    # ---------- Immediately offload heavy components ----------
    # Text encoder and VAE are only needed during generate(); keep UNet on GPU
    offload_to_cpu(model)

    # ---------- Setup ----------
    os.makedirs(args.output_dir, exist_ok=True)
    batch_size = args.batch_size

    inference_log = {
        "timestamp":    datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        "checkpoint":   args.checkpoint,
        "object_class": args.object_class,
        "defect_type":  args.defect_type,
        "results":      []
    }

    #  Dynamic Dataset Generation
    if args.total_images > 0 and args.data_dir and args.defect_type:
        # Get good samples, masks
        num_good, num_masks, good_dir, mask_dir = count_available_resources(
            args.data_dir, args.object_class, args.defect_type
        )
        if num_good == 0 or num_masks == 0:
            print("Error: Missing images or masks.") # No ref image or ref mask
            return

        generation_plan = calculate_generation_plan(num_good, num_masks, args.total_images) # list((good_idx, mask_idx, output_idx))
        good_files = sorted([f for f in os.listdir(good_dir) if f.endswith(('.png','.jpg','.jpeg'))])
        mask_files = sorted([f for f in os.listdir(mask_dir) if f.endswith('.png')])

        org_output_dir = os.path.join(args.output_dir, args.defect_type, 'org')
        defect_output_dir = os.path.join(args.output_dir, args.defect_type, 'defective')
        mask_output_dir = os.path.join(args.output_dir, args.defect_type, 'mask')
        os.makedirs(defect_output_dir, exist_ok=True)
        os.makedirs(org_output_dir, exist_ok=True)
        os.makedirs(mask_output_dir, exist_ok=True)

        prompt = f"A {args.object_class} with {model.get_token_for(args.object_class, args.defect_type)}"
        # print(f"Prompt: '{prompt}'")

        for good_idx, mask_idx, output_idx in tqdm(generation_plan, desc=f"Generating {args.defect_type}"):
            print(f"\n[{output_idx+1}/{len(generation_plan)}] {good_files[good_idx]}")

            # Load and crop mask
            image_np = np.array(Image.open(os.path.join(good_dir, good_files[good_idx])).convert("RGB"))
            mask_np  = np.array(Image.open(os.path.join(mask_dir, mask_files[mask_idx])).convert("L"))

            # minh: Crop crop the part of the image containing the defect, 
            #       keep the resolution for defects with size < 512, resize large defects to 512x512
            crop_img_np, crop_mask_np, x1, y1, x2, y2 = smart_crop_dynamic(image_np, mask_np) 

            # [0, 255] -> [0.0, 1.0] -> Normalize to [-1.0, 1.0]
            img_tensor  = transforms.Normalize([0.5]*3, [0.5]*3)(
                              transforms.ToTensor()(crop_img_np)
                          ).unsqueeze(0).to(device, dtype=dtype)
            
            mask_tensor = transforms.ToTensor()(crop_mask_np).unsqueeze(0).to(device, dtype=dtype)

            # --- Bring encoder/VAE to GPU just for generation ---
            reload_to_gpu(model, device, dtype)

            with torch.no_grad():
                best_sample, lpips_score = select_best_sample(
                    model, img_tensor, mask_tensor, prompt,
                    num_samples=args.num_samples,
                    steps=args.steps,
                    guidance_scale=args.guidance_scale,
                    device=device,
                    dtype=dtype,
                )

            # --- Offload again immediately after ---
            offload_to_cpu(model)

            # --- Save outputs ---
            prefix_org = os.path.join(org_output_dir, f"{output_idx:04d}")
            prefix_def = os.path.join(defect_output_dir, f"{output_idx:04d}")
            prefix_mask = os.path.join(mask_output_dir, f"{output_idx:04d}")
            result_np = paste_back(image_np, best_sample, x1, y1, x2, y2)
 
            Image.fromarray(image_np).save(f"{prefix_org}_original.png")
            Image.fromarray(result_np).save(f"{prefix_def}_generated.png")
            # save_image((best_sample.float() + 1) / 2, f"{prefix}_generated.png")
            # save_image((img_tensor.float() + 1) / 2,  f"{prefix}_original.png") # [-1, 1] -> [0, 1]
            save_image(mask_tensor.float(), f"{prefix_mask}_mask.png")

            inference_log["results"].append({
                "output_idx":  output_idx,
                "input_image": good_files[good_idx],
                "lpips_score": lpips_score,
            })

            # --- Cleanup ---
            del img_tensor, mask_tensor, best_sample
            flush()
    log_path = os.path.join(args.output_dir, "inference_log.json")
    with open(log_path, "w") as f:
        json.dump(inference_log, f, indent=4)
    print(f"\nInference log saved: {log_path}")

In [ ]:
DATA_DIR = "/kaggle/input/datasets/dahyuntw/curated-mvtec/curated_mvtec"

In [ ]:
args = argparse.Namespace(
    checkpoint    = "/kaggle/input/models/dahyuntw/ablation-bottle/pytorch/default/1/ablation_bottle/ablated_obj.pth",  # path to your saved model
    output_dir    = "/kaggle/working/generated",
    object_class  = "bottle",       # change to your class
    defect_type   = "poke",       # change to your defect type
    data_dir      = DATA_DIR,  # root dataset dir
    image_path    = None,
    image_dir     = None,
    num_samples   = 1, # 5 Candidates per image, choose the best one
    total_images  = 25, # 25 Number of variations to be generated
    steps         = 50,
    guidance_scale= 7.5,
    batch_size    = 2,
    use_compile   = False,          # keep False, torch.compile is slow on first run
    lora_alpha    = 16,
    lora_rank     = 8,
    dilate_mask   = False,
    mask_kernel_size = 3,
)

In [ ]:
leather_defects = ['broken_large', 'broken_small', 'contamination']

In [ ]:
for leather_def in leather_defects:
    args.defect_type = leather_def
    inference(args)

## 4. Benchmark

In [ ]:
import torch_fidelity
from pathlib import Path
import csv
import tempfile, shutil

In [ ]:
gen_dir = Path("/kaggle/input/datasets/dahyuntw/gen-results/2. results")

In [ ]:
def evaluate_benchmark(real_dir: str, gen_dir: str, device: str) -> dict:
    # Ref: https://github.com/devavratTomar/torch-fidelity/blob/37d34e7d64cb493e5fe10be99b1ff1a4ab1a3f64/torch_fidelity/metrics.py#L10
    metrics = torch_fidelity.calculate_metrics(
        input1=gen_dir,           # IS is computed on input1
        input2=real_dir,          # FID/KID compare input1 vs input2
        cuda=device,
        isc=False,                # Inception Score
        fid=True,                 # Fréchet Inception Distance
        kid=True,                 # Kernel Inception Distance
        kid_subset_size=min(      # KID subset — auto-shrink for small datasets
            100,
            len([f for f in os.listdir(real_dir)
                 if f.lower().endswith((".png", ".jpg", ".jpeg"))]),
            len([f for f in os.listdir(gen_dir)
                 if f.lower().endswith((".png", ".jpg", ".jpeg"))]),
        ),
        verbose=False,
    )

    return {
        "FID": metrics["frechet_inception_distance"],
        "KID": {
            "mean": metrics["kernel_inception_distance_mean"],
            "std":  metrics["kernel_inception_distance_std"],
        },
    }


In [ ]:
def img_to_tensor(img_bgr):
    """Convert BGR uint8 HxWxC → [-1,1] tensor for LPIPS."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    t = transforms.ToTensor()(img_rgb)  # [0,1]
    t = t * 2.0 - 1.0                   # [-1,1]
    return t.unsqueeze(0).to(device)


def load_images(folder):
    """Return sorted list of image paths in a folder."""
    return sorted([f for f in Path(folder).iterdir()
                   if f.suffix.lower() in IMG_EXTS])

def save_crop(img, mask, out_path, base_size=512):
    crop, _, _, _, _, _ = smart_crop_dynamic(img, mask, base_size)
    cv2.imwrite(out_path, crop)

    return crop

In [ ]:
obj_folders = [f for f in gen_dir.iterdir() if f.is_dir()]
all_results = []

for obj_path in obj_folders:
    obj = obj_path.name
    obj_results = []
    defect_folders = [f for f in obj_path.iterdir() if f.is_dir()]

    for defect_path in defect_folders:
        defect = defect_path.name
        gen_defect_folder = defect_path / "defective"
        mask_folder       = defect_path / "mask"
        org_defect_folder = defect_path / "org_defect"

        if not (gen_defect_folder.is_dir() and mask_folder.is_dir() and org_defect_folder.is_dir()):
            print(f"[Skip] Missing subfolders for {obj}/{defect}")
            continue

        IMG_EXTS = (".png", ".jpg", ".jpeg")
        gen_imgs = sorted([f for f in gen_defect_folder.iterdir() if f.suffix.lower() in IMG_EXTS])
        mask_imgs = sorted([f for f in mask_folder.iterdir()       if f.suffix.lower() in IMG_EXTS])
        org_imgs  = sorted([f for f in org_defect_folder.iterdir() if f.suffix.lower() in IMG_EXTS])

        n = len(org_imgs)
        if n == 0:
            print(f"[Skip] No org_defect images for {obj}/{defect}")
            continue

        print(f"Evaluating {obj}/{defect}  (gen={len(gen_imgs)}, mask={len(mask_imgs)}, org_defect={n})")

        # Create temp folders for cropped images
        tmp_gen = tempfile.mkdtemp()
        tmp_org = tempfile.mkdtemp()

        try:
            for i in range(n):
                gen_img  = cv2.imread(str(gen_imgs[i % len(gen_imgs)]))
                mask_img = cv2.imread(str(mask_imgs[i % len(mask_imgs)]), cv2.IMREAD_GRAYSCALE)
                org_img  = cv2.imread(str(org_imgs[i]))
    
                if gen_img is None or mask_img is None or org_img is None:
                    print(f"  [Warn] Could not read image at index {i}, skipping.")
                    continue
    
                gen_crop = save_crop(gen_img, mask_img, os.path.join(tmp_gen, f"{i:04d}.png"))
                org_crop = save_crop(org_img, mask_img, os.path.join(tmp_org, f"{i:04d}.png"))
                # show_tensors(img_to_tensor(org_crop), img_to_tensor(gen_crop), img_to_tensor(mask_img))
            scores = evaluate_benchmark(tmp_org, tmp_gen, device)
            row = {
                "object":   obj,
                "defect":   defect,
                "FID":      scores["FID"],
                "KID_mean": scores["KID"]["mean"],
                "KID_std":  scores["KID"]["std"],
            }
            obj_results.append(row)
            all_results.append(row)
        # print(row)

        except Exception as e:
            print(f"[Error] {obj}/{defect}: {e}")

        finally:
            shutil.rmtree(tmp_gen, ignore_errors=True)
            shutil.rmtree(tmp_org, ignore_errors=True)

    if obj_results:
        avg_row = {
            "object":   obj,
            "defect":   "AVERAGE",
            "FID":      sum(r["FID"]      for r in obj_results) / len(obj_results),
            "KID_mean": sum(r["KID_mean"] for r in obj_results) / len(obj_results),
            "KID_std":  sum(r["KID_std"]  for r in obj_results) / len(obj_results),
        }
        all_results.append(avg_row)
        print(f"\n[{obj}] Average — FID: {avg_row['FID']:.4f}  KID: {avg_row['KID_mean']:.6f}")

defect_rows = [r for r in all_results if r["defect"] != "AVERAGE"]
if defect_rows:
    overall = {
        "object":   "OVERALL",
        "defect":   "AVERAGE",
        "FID":      sum(r["FID"]      for r in defect_rows) / len(defect_rows),
        "KID_mean": sum(r["KID_mean"] for r in defect_rows) / len(defect_rows),
        "KID_std":  sum(r["KID_std"]  for r in defect_rows) / len(defect_rows),
    }
    all_results.append(overall)

csv_path = os.path.join('/kaggle/working/', "metrics_cropped.csv")
fieldnames = ["object", "defect", "FID", "KID_mean", "KID_std"]
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_results)
print(f"\nCSV saved to: {csv_path}")

## LPIPS

In [ ]:
import lpips

In [ ]:
RESULTS_DIR = r'/kaggle/input/datasets/dahyuntw/gen-results/2. results'
OUTPUT_CSV  = r'/kaggle/working/lpips_metrics.csv'
BASE_SIZE   = 512
IMG_EXTS    = ('.png', '.jpg', '.jpeg')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

loss_fn = lpips.LPIPS(net='vgg').to(device)
loss_fn.eval()

In [ ]:
results_root = Path(RESULTS_DIR)
all_rows = []
obj_folders = sorted([f for f in results_root.iterdir() if f.is_dir()])

for obj_path in obj_folders:
    obj_name = obj_path.name
    obj_lpips_list = []

    defect_folders = sorted([f for f in obj_path.iterdir() if f.is_dir()])
    print(defect_folders)

    for defect_path in defect_folders:
        defect_name = defect_path.name

        gen_defective_dir = defect_path / 'defective'
        mask_dir          = defect_path / 'mask'
        org_defect_dir    = defect_path / 'org_defect'

        if not (gen_defective_dir.is_dir() and mask_dir.is_dir() and org_defect_dir.is_dir()):
            print(f'[Skip] Missing folders for {obj_name}/{defect_name}')
            continue

        gen_imgs  = load_images(gen_defective_dir)
        mask_imgs = load_images(mask_dir)
        org_imgs  = load_images(org_defect_dir)

        # Anchor loop to org_defect (real images, smallest set)
        n = len(org_imgs)
        if n == 0:
            print(f'[Skip] No org_defect images for {obj_name}/{defect_name}')
            continue

        print(f'Evaluating {obj_name}/{defect_name}  '
              f'(gen={len(gen_imgs)}, mask={len(mask_imgs)}, org_defect={n})')

        pair_scores = []

        for i in range(n):
            gen_path  = gen_imgs[i % len(gen_imgs)]
            mask_path = mask_imgs[i % len(mask_imgs)]
            org_path  = org_imgs[i]

            gen_img  = cv2.imread(str(gen_path))
            mask_img = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            org_img  = cv2.imread(str(org_path))

            if gen_img is None or mask_img is None or org_img is None:
                print(f'  [Warn] Could not read image at index {i}, skipping.')
                continue

            # Both images cropped with the same mask (same scene, same defect location)
            gen_crop, _, _, _, _, _ = smart_crop_dynamic(gen_img, mask_img, BASE_SIZE)
            org_crop, _, _, _, _, _ = smart_crop_dynamic(org_img, mask_img, BASE_SIZE)

            t_gen = img_to_tensor(gen_crop)
            t_org = img_to_tensor(org_crop)
            
            # show_tensors(t_org, t_gen, img_to_tensor(mask_img))
            with torch.no_grad():
                score = loss_fn(t_gen, t_org).item()

            pair_scores.append(score)

        if pair_scores:
            mean_lpips = np.mean(pair_scores)
            std_lpips  = np.std(pair_scores)
            row = {
                'object':     obj_name,
                'defect':     defect_name,
                'n_pairs':    len(pair_scores),
                'LPIPS_mean': round(mean_lpips, 6),
                'LPIPS_std':  round(std_lpips,  6),
            }
            all_rows.append(row)
            obj_lpips_list.append(mean_lpips)
            print(f'  → LPIPS mean={mean_lpips:.4f}  std={std_lpips:.4f}  (n={len(pair_scores)})')

    # Per-object average
    if obj_lpips_list:
        avg = np.mean(obj_lpips_list)
        all_rows.append({
            'object':     obj_name,
            'defect':     'AVERAGE',
            'n_pairs':    '',
            'LPIPS_mean': round(avg, 6),
            'LPIPS_std':  '',
        })
        print(f'\n[{obj_name}] Average LPIPS = {avg:.4f}\n')

# Overall average
defect_rows = [r for r in all_rows if r['defect'] != 'AVERAGE']
if defect_rows:
    overall = np.mean([r['LPIPS_mean'] for r in defect_rows])
    all_rows.append({
        'object':     'OVERALL',
        'defect':     'AVERAGE',
        'n_pairs':    '',
        'LPIPS_mean': round(overall, 6),
        'LPIPS_std':  '',
    })
    print(f'\nOVERALL Average LPIPS = {overall:.4f}')

print('\nDone.')

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
fieldnames = ['object', 'defect', 'n_pairs', 'LPIPS_mean', 'LPIPS_std']

with open(OUTPUT_CSV, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)

print(f'CSV saved to: {OUTPUT_CSV}')

## IC-LPIPS

In [ ]:
import cv2

In [ ]:
loss_fn = lpips.LPIPS(net='alex').to(device)
loss_fn.eval()

In [ ]:
def ic_lpips(gen_imgs, org_imgs, loss_fn, device):
    l = len(org_imgs) // 3  # one cluster per real image

    # Preload real image tensors
    org_tensors = []
    for org_path in org_imgs:
        img = cv2.imread(str(org_path))
        # print(img)
        org_tensors.append(img_to_tensor(img).to(device))

    # Assign each generated image to nearest real image cluster
    clusters = [[] for _ in range(l)]
    for gen_path in gen_imgs:
        img = cv2.imread(str(gen_path))
        t_gen = img_to_tensor(img)
        min_dist, best_k = float('inf'), 0
        with torch.no_grad():
            for k, t_org in enumerate(org_tensors):
                dist = loss_fn(t_gen, t_org).item()
                if dist < min_dist:
                    min_dist, best_k = dist, k
        clusters[best_k].append(t_gen)

    # Pairwise LPIPS within each cluster
    cluster_scores = []
    for k, cluster in enumerate(clusters):
        if len(cluster) < 2:
            continue  # need at least 2 to compute pairwise
        dists = []
        for i in range(len(cluster)):
            for j in range(i + 1, len(cluster)):
                with torch.no_grad():
                    dists.append(loss_fn(cluster[i], cluster[j]).item())
        cluster_scores.append(np.mean(dists))

    return np.mean(cluster_scores) if cluster_scores else float('nan')

In [ ]:
obj_folders = [f for f in gen_dir.iterdir() if f.is_dir()]
all_results = []

for obj_path in obj_folders:
    obj = obj_path.name
    obj_results = []
    defect_folders = [f for f in obj_path.iterdir() if f.is_dir()]

    for defect_path in defect_folders:
        defect = defect_path.name
        gen_defect_folder = defect_path / "defective"
        mask_folder       = defect_path / "mask"
        org_defect_folder = defect_path / "org_defect"

        if not (gen_defect_folder.is_dir() and mask_folder.is_dir() and org_defect_folder.is_dir()):
            print(f"[Skip] Missing subfolders for {obj}/{defect}")
            continue

        IMG_EXTS = (".png", ".jpg", ".jpeg")
        gen_imgs  = sorted([f for f in gen_defect_folder.iterdir() if f.suffix.lower() in IMG_EXTS])
        mask_imgs = sorted([f for f in mask_folder.iterdir()       if f.suffix.lower() in IMG_EXTS])
        org_imgs  = sorted([f for f in org_defect_folder.iterdir() if f.suffix.lower() in IMG_EXTS])

        n = len(gen_imgs)
        if n == 0:
            print(f"[Skip] No generated images for {obj}/{defect}")
            continue

        print(f"Evaluating {obj}/{defect}  (gen={len(gen_imgs)}, mask={len(mask_imgs)}, org_defect={len(org_imgs)})")

        tmp_gen = tempfile.mkdtemp()
        tmp_org = tempfile.mkdtemp()

        try:
            for i in range(n):
                gen_img  = cv2.imread(str(gen_imgs[i % len(gen_imgs)]))
                mask_img = cv2.imread(str(mask_imgs[i % len(mask_imgs)]), cv2.IMREAD_GRAYSCALE)
                save_crop(gen_img, mask_img, os.path.join(tmp_gen, f"{i:04d}.png"))
                if i < len(org_imgs):
                    org_img = cv2.imread(str(org_imgs[i]))
                    save_crop(org_img, mask_img, os.path.join(tmp_org, f"{i:04d}.png"))

            gen_img_paths = sorted([p for p in Path(tmp_gen).iterdir() if p.is_file()])
            org_img_paths = sorted([p for p in Path(tmp_org).iterdir() if p.is_file()])

            ic_score = ic_lpips(gen_img_paths, org_img_paths, loss_fn, device)
            print(f"  IC-LPIPS = {ic_score:.4f}")

            row = {
                "object":   obj,
                "defect":   defect,
                "IC_LPIPS": round(float(ic_score), 6) if not np.isnan(ic_score) else "",
            }
            obj_results.append(row)
            all_results.append(row)

        except Exception as e:
            print(f"[Error] {obj}/{defect}: {e}")

        finally:
            shutil.rmtree(tmp_gen, ignore_errors=True)
            shutil.rmtree(tmp_org, ignore_errors=True)

    # Per-object average
    if obj_results:
        valid = [r["IC_LPIPS"] for r in obj_results if r["IC_LPIPS"] != ""]
        avg = np.mean(valid) if valid else ""
        all_results.append({
            "object":   obj,
            "defect":   "AVERAGE",
            "IC_LPIPS": round(float(avg), 6) if avg != "" else "",
        })
        print(f"\n[{obj}] Average IC-LPIPS = {avg:.4f}\n")

# Overall average
defect_rows = [r for r in all_results if r["defect"] != "AVERAGE" and r["IC_LPIPS"] != ""]
if defect_rows:
    overall = np.mean([r["IC_LPIPS"] for r in defect_rows])
    all_results.append({
        "object":   "OVERALL",
        "defect":   "AVERAGE",
        "IC_LPIPS": round(float(overall), 6),
    })
    print(f"\nOVERALL Average IC-LPIPS = {overall:.4f}")

# Write CSV
csv_path = os.path.join('/kaggle/working/', "ic_lpips_metrics.csv")
fieldnames = ["object", "defect", "IC_LPIPS"]
with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_results)

print(f"\nCSV saved to: {csv_path}")